In [ ]:
import pickle

with open('KD/datasets/selected_questions_with_answers_arc_part_1_one_sentence_reason.pickle', 'rb') as f:
    arc_data = pickle.load(f)
with open('KD/datasets/selected_questions_with_answers_pub_part_1_one_sentence_reason.pickle', 'rb') as f:
    pub_data = pickle.load(f)
with open('KD/datasets/selected_questions_with_answers_popqa_part_1_one_sentence_reason.pickle', 'rb') as f:
    popqa_data = pickle.load(f)

In [ ]:
import json
# building inputs and outputs
# prompt_template = "Question: {question} Context: {context} \n\nIs The question answered by the context?"
prompt_template = "Question: {question} Context: {context} \n\nTo what degree can we confidently assert that the context provides a clear and accurate answer to the question? only answer in one word: high, medium or low"
pub_prompt_template = "Question: Is the following statement correct? {question} Context: {context} \n\nTo what degree can we confidently assert that the context provides a clear and accurate answer to the question? only answer in one word: high, medium or low"

inputs = []
outputs = []
# for ARC
for d in arc_data:
  for context in d[1]:
    for answer in d[2]:
      if context['id'] == answer['document_id']:
        if answer['answer']['rating'] != "high" and answer['answer']['rating'] != "medium" and answer['answer']['rating'] != "low" :
          continue
        inputs.append(prompt_template.format(question=d[0],context=context['text']))
        outputs.append(json.dumps(answer['answer']))
for d in pub_data:
  for context in d[1]:
    for answer in d[2]:
      if context['id'] == answer['document_id']:
        if answer['answer']['rating'] != "high" and answer['answer']['rating'] != "medium" and answer['answer']['rating'] != "low" :
          continue
        inputs.append(pub_prompt_template.format(question=d[0],context=context['text']))
        outputs.append(json.dumps(answer['answer']))
for d in popqa_data:
  if d[2]['rating'] != "high" and d[2]['rating'] != "medium" and d[2]['rating'] != "low" :
    continue
  inputs.append(prompt_template.format(question=d[0],context=d[1]))
  outputs.append(json.dumps(d[2]))


## Loading Reflection Prompt Data

In [ ]:
reflection_template = """
We have a Question and a Context. Previously you were asked:
'To what degree can we confidently assert that the context provides a clear and accurate answer to the question?'
Possible first-pass ratings were: high, medium, low. For this item you previously selected: "medium".

Now you will re-evaluate this same item more carefully. Read the original justification below, then re-assess.

Original justification:
\"\"\"{original_reason}\"\"\"

Question:
{question}

Context:
{context}

Important instructions:
- Re-rate using only one of these two labels: "high" or "low" (do NOT output 'medium').
- If you are unsure or the context is ambiguous, prefer "low".
- Output EXACTLY one JSON object and nothing else. The JSON must include:

{{
  "new_rating": "<either 'high' or 'low'>",
  "justification": "<one or two short sentences>"
}}
"""

In [ ]:
with open('KD/datasets/reflection_results_medium_cases_r1_new_2_deduplicated.pickle', 'rb') as f:
    reflection_data = pickle.load(f)

In [ ]:
reflection_inputs = []
reflection_outputs = []
high_counter = 0
for d in reflection_data:
  reflection_inputs.append(reflection_template.format(original_reason=d.get("original_reason"),question=d.get("question"),context=d.get("context")))
  reflection_outputs.append(json.dumps({"new_rating": d.get("new_rating"), "justification": d.get("justification")}))
  if d.get("new_rating").lower() == "high":
    high_counter += 1

print(high_counter)
# Addding reflections to inputs and outputs
inputs += reflection_inputs
outputs += reflection_outputs

478


In [ ]:
# !pip install git+https://github.com/huggingface/transformers.git
!pip install peft accelerate datasets

In [ ]:
print(f"Inputs type: {[type(i) for i in inputs]}")
print(f"Outputs type: {[type(o) for o in outputs]}")

Inputs type: [<class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str

In [ ]:
# Verify dataset size
print(f"Number of inputs: {len(inputs)}")
print(f"Number of outputs: {len(outputs)}")
print(f"Sample input: {inputs[0]}")
print(f"Sample output: {outputs[0]}")

Number of inputs: 13591
Number of outputs: 13591
Sample input: Question: Which of the following best explains how stems transport water to other parts of the plant? A) through a chemical called chlorophyll B) by using photosynthesis C) through a system of tubes D) by converting water to food Context: transpiration and CO exchange. In vascular plants, water is acquired from the soil by roots and transported via the xylem to aerial portions of the plant. Water evaporation from the aerial surfaces of the plant is controlled by a waterproof covering of cuticle. Gas exchange with the atmosphere is controlled by stomata, which can open and close to control water loss, and diffusion of carbon dioxide to the chloroplasts takes place in intercellular spaces between chlorenchyma cells in the stem or in the mesophyll tissue of the leaf. The antonym of homoiohydry is poikilohydry, a condition in which plant water 

To what degree can we confidently assert that the context provides a clear and accu

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 12.1 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Create the dataset
data_dict = {"inputs": inputs, "targets": outputs}
dataset = Dataset.from_dict(data_dict)

# Combine inputs and targets into a single text field
def format_example(example):
    return {
        "text": f"{example['inputs']} \n\nAnswer: {example['targets']}"
    }
dataset = dataset.map(format_example)

# Split dataset into train and eval (80-20 split)
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

# Load TinyLLaMA model and tokenizer
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Apply LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Define training arguments
training_args = SFTConfig(
    output_dir="fine-tuned-models/qwen_lora_finetuned_batch8_one_sentence_reasoning_correctformat_plus_reflection_r1",
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=1000,
    eval_steps=100,
    logging_dir="./logs",
    logging_steps=100,
    per_device_train_batch_size=5,
    per_device_eval_batch_size=5,
    gradient_accumulation_steps=4,
    num_train_epochs=12,
    learning_rate=2e-4,
    weight_decay=0.1,
    warmup_steps=100,
    fp16=True,
    report_to="tensorboard",
    save_total_limit=2,
    load_best_model_at_end=True,
    optim="adamw_torch",
    max_grad_norm=1.0,
)

# Define the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    # max_seq_length=512,  # Set max sequence length for tokenization
)


In [ ]:
# Fine-tune the model
trainer.train()

# Save the fine-tuned model
trainer.save_model("fine-tuned-models/qwen_lora_finetuned_batch8_one_sentence_reasoning_correctformat_plus_reflection_r1")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.069000,1.515300,1.514522,605199.000000,0.673112
200,1.500900,1.475078,1.485480,1196821.000000,0.679535
300,1.464100,1.459149,1.456611,1796121.000000,0.681540
400,1.439000,1.449082,1.446553,2402510.000000,0.683524
500,1.444500,1.442632,1.444432,3000707.000000,0.684810
600,1.443300,1.436572,1.440001,3596974.000000,0.685659
700,1.409500,1.433611,1.422523,4198864.000000,0.686103
800,1.407200,1.425374,1.423461,4801182.000000,0.687630
900,1.407300,1.421301,1.414827,5395935.000000,0.688455
1000,1.404800,1.414976,1.418561,5993355.000000,0.689671
